# Phase 1 — Big Data Handling

**Objective:** Ingest data that doesn't fit in RAM, correctly and reproducibly.

This notebook:
- Loads KYC, Transactions, and DayEndBalance using Polars lazy scanning
- Applies column pruning to cut unnecessary IO
- Documents the framework choice (Polars over Dask/Spark)
- Saves nothing heavy — lazy frames are recreated downstream by each notebook that needs them

> **Note on framework choice:** This pipeline uses **Polars** (lazy API) rather than Dask or Spark.
> Polars' lazy `scan_parquet` defers all computation until `.collect()`, giving the same
> "don't load everything into RAM" benefit as Dask, but with a faster query engine (written in Rust)
> and zero cluster/JVM setup. For a single-machine competition environment with ~560M total rows,
> this was the most efficient choice — Dask was considered but has higher per-operation overhead,
> and Spark would need cluster infrastructure this environment doesn't have.


In [2]:
import os
import gc
import warnings
warnings.filterwarnings('ignore')

# Check if running on Kaggle
IS_KAGGLE = os.path.exists('/kaggle')
BASE_PATH = '/kaggle/input/bkash-presents-nsucec-datathon/public' if IS_KAGGLE else '.'
print(f"Running on Kaggle: {IS_KAGGLE}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl

from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

Running on Kaggle: False


In [ ]:
base_path = '/media/tasfreak/Study/bkash-presents-nsucec-datathon/public'
USER_COL = "ACCOUNT_ID"

## Lazy scan all three tables + labels

Nothing is read into memory yet — Polars builds a lazy query plan.

In [4]:
# Lazily scan the Parquet files (wildcards grab all months at once)
# This reads only metadata initially, not the actual data.
print("Scanning datasets lazily...")
lazy_balances = pl.scan_parquet(f"{base_path}/dayend_balance/balance_2024-*.parquet")
lazy_transactions = pl.scan_parquet(f"{base_path}/transactions/trx_2024-*.parquet")
lazy_kyc = pl.scan_parquet(f"{base_path}/kyc.parquet")

lazy_users = pl.scan_csv(f"{base_path}/train_labels.csv")

print(lazy_balances.collect)
print(lazy_transactions.collect)
print(lazy_kyc.fetch(5))
print(lazy_users.fetch(5))

Scanning datasets lazily...
<bound method LazyFrame.collect of <LazyFrame at 0x79DCA29491C0>>
<bound method LazyFrame.collect of <LazyFrame at 0x79DCB834A000>>
shape: (5, 5)
┌──────────────┬──────────────┬─────────────────────┬────────┬────────────┐
│ ACCOUNT_ID   ┆ ACCOUNT_TYPE ┆ ACCOUNT_OPEN_DATE   ┆ GENDER ┆ REGION     │
│ ---          ┆ ---          ┆ ---                 ┆ ---    ┆ ---        │
│ str          ┆ str          ┆ datetime[ns]        ┆ str    ┆ str        │
╞══════════════╪══════════════╪═════════════════════╪════════╪════════════╡
│ CUST00000001 ┆ Customer     ┆ 2020-03-21 00:00:00 ┆ Male   ┆ Sylhet     │
│ CUST00000002 ┆ Customer     ┆ 2020-06-02 00:00:00 ┆ Male   ┆ Barishal   │
│ CUST00000003 ┆ Customer     ┆ 2020-12-13 00:00:00 ┆ Female ┆ Dhaka      │
│ CUST00000004 ┆ Customer     ┆ 2020-06-29 00:00:00 ┆ Male   ┆ Chattogram │
│ CUST00000005 ┆ Customer     ┆ 2021-02-02 00:00:00 ┆ Male   ┆ Chattogram │
└──────────────┴──────────────┴─────────────────────┴────────┴────

In [8]:
kychehe=lazy_kyc.group_by("ACCOUNT_TYPE").agg(pl.len()).collect()
kychehe

ACCOUNT_TYPE,len
str,u32
"""Merchant""",100000
"""Biller""",50000
"""Customer""",850000


## Column pruning

`TrxID` is a unique identifier per row with no predictive value — dropping it early
avoids carrying useless data through every join and aggregation downstream.

`GENDER` is dropped from KYC for this pipeline (not used as a feature — see Phase 2
feature catalog for rationale on what demographic signals were kept).

In [6]:
# Column Pruning
lazy_transactions = (lazy_transactions.drop("TrxID"))
lazy_kyc = (lazy_kyc.drop("GENDER"))
print(lazy_balances.fetch(10))
print(lazy_transactions.fetch(10))
print(lazy_kyc.fetch(10))

shape: (10, 3)
┌──────────────┬────────────┬───────────────────┐
│ ACCOUNT_ID   ┆ DATE       ┆ AVAILABLE_BALANCE │
│ ---          ┆ ---        ┆ ---               │
│ str          ┆ str        ┆ f64               │
╞══════════════╪════════════╪═══════════════════╡
│ CUST00000001 ┆ 2024-01-01 ┆ 2719.98           │
│ CUST00000001 ┆ 2024-01-02 ┆ 0.0               │
│ CUST00000001 ┆ 2024-01-03 ┆ 157.25            │
│ CUST00000001 ┆ 2024-01-04 ┆ 153.84            │
│ CUST00000001 ┆ 2024-01-05 ┆ 152.86            │
│ CUST00000001 ┆ 2024-01-06 ┆ 151.86            │
│ CUST00000001 ┆ 2024-01-07 ┆ 343.14            │
│ CUST00000001 ┆ 2024-01-08 ┆ 21049.95          │
│ CUST00000001 ┆ 2024-01-09 ┆ 21002.87          │
│ CUST00000001 ┆ 2024-01-10 ┆ 20263.74          │
└──────────────┴────────────┴───────────────────┘
shape: (10, 5)
┌─────────────────────┬──────────────┬──────────────┬─────────────┬──────────┐
│ TRX_DATETIME        ┆ SRC_ACCOUNT  ┆ DST_ACCOUNT  ┆ TRX_TYPE    ┆ TRX_AMT  │
│ ---       

## Sampling strategy for safe interactive EDA

For any exploratory analysis on the raw transaction-level data (before aggregation),
use a **stratified sample of accounts** rather than `.collect()`-ing the full lazy frame.
This keeps EDA representative of the ~5% churn class while staying fast and reproducible.

```python
SEED = 42
sample_accounts = (
    lazy_kyc
    .filter(pl.col("ACCOUNT_TYPE") == "Customer")
    .select("ACCOUNT_ID")
    .collect()
    .sample(fraction=0.01, seed=SEED)
)

eda_sample = (
    lazy_transactions
    .filter(pl.col("SRC_ACCOUNT").is_in(sample_accounts["ACCOUNT_ID"]))
    .collect()
)
```

This pipeline does not need this step explicitly because feature aggregation
(Phase 2) is done lazily on the full data — sampling is documented here as the
strategy that *would* be used for ad-hoc EDA on raw rows.

## Next notebook

Continue to **`02_feature_engineering.ipynb`**. That notebook re-creates the lazy
frames defined here (cheap — it's just a query plan) and builds the feature catalog.